In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import TRAIN_PATH, TEXT_COLUMNS
from src.preprocessing import preprocess_text_columns
from src.feature_engineering import add_sentiment_features, add_word_count_features

In [ ]:
df = pd.read_csv(TRAIN_PATH)
print(df.head())

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all"))

In [ ]:
# target distribution
target_counts = df["fraudulent"].value_counts(normalize=True) * 100

ax = target_counts.plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Fraudulent")
plt.ylabel("Percentage")

for i, value in enumerate(target_counts.values):
    ax.text(i, value + 1, f"{value:.1f}%", ha="center", fontweight="bold")

plt.xticks(ticks=[0, 1], labels=["Real", "Fraud"], rotation=0)
plt.show()

In [ ]:
# does null count correlate with the target? 
df_eda = df.copy()

# count missing values in each row
df_eda["nullCount"] = df_eda.isnull().sum(axis=1)

df_eda["is_department_null"] = df_eda["department"].isnull().astype(int)
df_eda["is_salary_null"] = df_eda["salary_range"].isnull().astype(int)


# plot fraud frequency by null count 
df_plot = df_eda[["fraudulent", "nullCount"]].copy()
df_plot["fraudulent"] = df_plot["fraudulent"].replace({0: "Real", 1: "Fraud"})

sns.kdeplot(
    data=df_plot,
    x="nullCount",
    hue="fraudulent",
    fill=True,
    alpha=0.4,
    linewidth=1.5
)

plt.title("Density Plot of Null Count")
plt.xlabel("Number of Missing Values in Row")
plt.ylabel("Density")
plt.show()

In [ ]:
# preprocess text for visualization 
df_text = df[["fraudulent"] + TEXT_COLUMNS].copy()
df_text = preprocess_text_columns(df_text, TEXT_COLUMNS)

df_text.head()

In [ ]:
def plot_sentiment(df, sentiment_column, title):
    plot_df = df[["fraudulent", sentiment_column]].copy()

    plot_df["fraudulent"] = plot_df["fraudulent"].replace({
        0: "Real",
        1: "Fraud"
    })

    plot_df[sentiment_column] = plot_df[sentiment_column].replace({
        0: "Negative",
        1: "Neutral",
        2: "Positive"
    })

    conditional_probabilities = pd.crosstab(
        plot_df["fraudulent"],
        plot_df[sentiment_column],
        normalize="index"
    )

    display(conditional_probabilities)

    ax = conditional_probabilities.plot(kind="bar")
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Percentage")
    plt.xticks(rotation=0)

    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[f"{value:.1%}" for value in container.datavalues],
            fontsize=10
        )

    plt.show()

In [ ]:
for column in TEXT_COLUMNS:
    sentiment_column = f"{column}_sentiment"

    plot_sentiment_by_target(
        df=df_text,
        sentiment_column=sentiment_column,
        title=f"Conditional Probability of {column} Sentiment"
    )

In [ ]:
# word count 
def plot_word_count_density(df, word_count_column, title):
    plot_df = df[["fraudulent", word_count_column]].copy()

    plot_df["fraudulent"] = plot_df["fraudulent"].replace({
        0: "Real",
        1: "Fraud"
    })

    sns.kdeplot(
        data=plot_df,
        x=word_count_column,
        hue="fraudulent",
        fill=True,
        alpha=0.4,
        linewidth=1.5
    )

    plt.title(title)
    plt.xlabel("Word Count")
    plt.ylabel("Density")
    plt.show()

In [ ]:
for column in TEXT_COLUMNS:
    word_count_column = f"{column}_word_count"

    plot_word_count_density(
        df=df_text,
        word_count_column=word_count_column,
        title=f"Density Plot of {column} Word Count"
    )